In [ ]:
# Cell 1 — Imports
import oracledb
import pandas as pd
import os
import datetime
from openpyxl import load_workbook
from openpyxl.styles import (Font, PatternFill, Alignment, Border, Side)
from openpyxl.utils import get_column_letter

In [ ]:
# Cell 2 — Configuration
# ── Database credentials ──
PERF_DB_USER     = "mkeita9"
PERF_DB_PASSWORD = "Support_123"
PERF_DB_DSN      = "FSMP.worldbank.org"

# ── Reporting date — only change this line ──
AS_OF_DATE = "20260724"

# ── Output ──
OUTPUT_DIR  = r"C:\Users\mkeita9\OneDrive - WBG\1. Work Documents\4. Projects & Initiatives\Murex_Testing_LAM_Cash\09_PLSummary\02_Input_Data\3_Summit_Performance_Exports"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"Summit_Performance_Trade_PnL_Report_{AS_OF_DATE}.xlsx")

# ── SQL Query ──
SQL_QUERY = f"""
SELECT
    t.TRADE_ID,
    p.isin,
    t.ASSET_TYPE_CODE,
    t.book_code,
    SUM(t.INCOME_BOOK_CCY_AMT)        AS DTD,
    SUM(t.MARKET_VALUE_BOOK_CCY_AMT)  AS MV,
    SUM(t.accrual_AMT)                AS AI,
    SUM(t.CURR_NTL_BOOK_CCY_AMT)      AS NOTIONAL_USD,
    SUM(t.NOTIONAL_BEG_AMT)           AS NOTIONAL,
    TO_CHAR(t.AS_OF_DATE, 'YYYYMMDD') AS AS_OF_DATE

FROM CURR_TRADE_PERF_V t,
     (
        SELECT *
        FROM PORTFOLIO_DEFN_MV
        WHERE maturity_dt >=  TO_DATE('{AS_OF_DATE}', 'YYYYMMDD')
          AND TRIM(isin) NOT IN ('000000000010', '000000000000')
     ) p

WHERE t.trade_id          = p.trade_id(+)
  AND t.security_id       = p.security_id(+)
  AND t.asset_type_code   = p.asset_type_code(+)
  AND t.book_code         = p.book_code(+)
  AND t.PORTFOLIO_TYPE_CODE = p.PORTFOLIO_TYPE_CODE(+)
  AND t.AS_OF_DATE        = TO_DATE('{AS_OF_DATE}', 'YYYYMMDD')
 -- AND t.BOOK_CODE         = 'P1'
  AND t.PORTFOLIO_TYPE_CODE = 'A'

GROUP BY
    t.TRADE_ID,
    p.isin,
    t.ASSET_TYPE_CODE,
    t.book_code,
    TO_CHAR(t.AS_OF_DATE, 'YYYYMMDD')

ORDER BY
    p.isin, t.TRADE_ID
"""


In [ ]:
# Cell 3 — Display function
def display_df(df):
    return df.style.set_properties(**{
        'white-space': 'nowrap',
        'font-size': '11px',
        'text-align': 'left',
        'border': '1px solid lightgrey',
        'color': 'black'
    }).set_table_styles([
        {
            'selector': 'thead th',
            'props': [
                ('white-space', 'nowrap'),
                ('font-size', '11px'),
                ('font-weight', 'bold'),
                ('background-color', '#4472C4'),
                ('color', 'white'),
                ('text-align', 'center'),
                ('border', '1px solid lightgrey')
            ]
        },
        {
            'selector': 'tbody tr:nth-child(even)',
            'props': [('background-color', '#f2f2f2'), ('color', 'black')]
        },
        {
            'selector': 'tbody tr:nth-child(odd)',
            'props': [('background-color', 'white'), ('color', 'black')]
        },
        {
            'selector': 'tbody tr:hover',
            'props': [('background-color', '#d6e4f0'), ('color', 'black')]
        }
    ])

In [ ]:
#Cell 4 — Connect and execute query
try:
    print("Connecting to Summit Performance database (FSMP)...")

    oracledb.init_oracle_client(config_dir=r"C:\oracle\network\admin")  # ← update if needed

    connection = oracledb.connect(
        user     = PERF_DB_USER,
        password = PERF_DB_PASSWORD,
        dsn      = PERF_DB_DSN
    )
    print(f"Connected successfully. Oracle version: {connection.version}")

    cursor = connection.cursor()

    # ── Set default schema ──
    cursor.execute("ALTER SESSION SET CURRENT_SCHEMA = TREASPERF")
    print("Schema set to TREASPERF.")

    # ── Execute main query ──
    print("Executing query...")
    cursor.execute(SQL_QUERY)

    # ── Fetch results into DataFrame ──
    columns = [col[0] for col in cursor.description]
    rows    = cursor.fetchall()
    df      = pd.DataFrame(rows, columns=columns)

    print(f"Total Rows    : {len(df)}")
    print(f"Total Columns : {len(df.columns)}")

except oracledb.DatabaseError as e:
    error, = e.args
    print(f"Database error code    : {error.code}")
    print(f"Database error message : {error.message}")

except Exception as e:
    print(f"Unexpected error: {type(e).__name__}: {e}")

finally:
    try:
        cursor.close()
        connection.close()
        print("Connection closed.")
    except:
        pass


In [ ]:
def clean_trade_id(val):
    if val is None:
        return val
    val = str(val).strip()
    if val.lstrip("0").isdigit():
        return int(val)   # removes leading zeros AND returns as integer
    return val            # keeps non-numeric as is e.g. 'SPINTAUDTAX'

df["TRADE_ID"] = df["TRADE_ID"].apply(clean_trade_id)


In [ ]:
# Cell 5 — Save to professionally formatted Excel
# ── Create output directory if it does not exist ──
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Save raw DataFrame first ──
df.to_excel(OUTPUT_FILE, index=False, sheet_name="Trade PnL Report")

# ── Load workbook for formatting ──
wb = load_workbook(OUTPUT_FILE)
ws = wb.active

# ── Define styles ──
header_font  = Font(name="Calibri", bold=True, color="FFFFFF", size=11)
header_fill  = PatternFill(fill_type="solid", fgColor="4472C4")
header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
data_font    = Font(name="Calibri", size=10)
data_align   = Alignment(horizontal="left", vertical="center")
number_align = Alignment(horizontal="right", vertical="center")
alt_fill     = PatternFill(fill_type="solid", fgColor="EEF2FA")
thin_border  = Border(
    left   = Side(style="thin", color="D9D9D9"),
    right  = Side(style="thin", color="D9D9D9"),
    top    = Side(style="thin", color="D9D9D9"),
    bottom = Side(style="thin", color="D9D9D9")
)

# ── Numeric columns ──
numeric_cols = ["DTD", "MV", "NOTIONAL_USD", "NOTIONAL"]

# ── Format header row ──
for cell in ws[1]:
    cell.font      = header_font
    cell.fill      = header_fill
    cell.alignment = header_align
    cell.border    = thin_border
ws.row_dimensions[1].height = 30

# ── Format data rows ──
for row_idx, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row), start=2):
    fill = alt_fill if row_idx % 2 == 0 else PatternFill()
    for cell in row:
        cell.font   = data_font
        cell.fill   = fill
        cell.border = thin_border
        if cell.column_letter in [
            get_column_letter(df.columns.get_loc(c) + 1)
            for c in numeric_cols if c in df.columns
        ]:
            cell.number_format = '#,##0.00'
            cell.alignment     = number_align
        else:
            cell.alignment = data_align

# ── Auto-fit column widths ──
for col in ws.columns:
    max_length = 0
    col_letter = col[0].column_letter
    for cell in col:
        if cell.value:
            max_length = max(max_length, len(str(cell.value)))
    ws.column_dimensions[col_letter].width = min(max_length + 4, 40)

# ── Freeze header row ──
ws.freeze_panes = "A2"

# ── Add totals row ──
last_row   = ws.max_row + 1
total_fill = PatternFill(fill_type="solid", fgColor="4472C4")

for col_idx, col_name in enumerate(df.columns, start=1):
    cell        = ws.cell(row=last_row, column=col_idx)
    cell.font   = Font(name="Calibri", bold=True, color="FFFFFF", size=10)
    cell.fill   = total_fill
    cell.border = thin_border
    if col_name in numeric_cols:
        col_letter         = get_column_letter(col_idx)
        cell.value         = f"=SUM({col_letter}2:{col_letter}{last_row - 1})"
        cell.number_format = '#,##0.00'
        cell.alignment     = number_align
    elif col_idx == 1:
        cell.value     = "TOTAL"
        cell.alignment = Alignment(horizontal="left", vertical="center")

# ── Save final workbook ──
wb.save(OUTPUT_FILE)
print(f"Excel report saved to: {OUTPUT_FILE}")
